<a href="https://colab.research.google.com/github/O-2wice/correctness-aware-nl-query-translation-ocel/blob/main/notebooks/03_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 03 Results — Expanded Benchmark Analysis

**Purpose:** Compare all four methods on the current 74-question development split, with the 46-question held-out test summary included for reporting.
This notebook regenerates the comparison figures, repair-loop analysis, and failure-taxonomy outputs from the current expanded-benchmark CSV artifacts.

**Canonical inputs:**
- `outputs/reports/baseline_b1_dev.csv`
- `outputs/reports/baseline_b2_dev.csv`
- `outputs/reports/baseline_b3_dev.csv`
- `outputs/reports/pipeline_dev.csv`
- `outputs/reports/method_comparison.csv`

**Outputs:**
- `outputs/figures/comparison_bar.pdf`
- `outputs/figures/denacc_by_class.pdf`
- `outputs/figures/denacc_by_difficulty.pdf`
- `outputs/figures/pipeline_status.pdf`
- `outputs/figures/latency.pdf`
- `outputs/figures/join_hallucination.pdf`
- `outputs/reports/dev_summary.csv`


In [ ]:
from pathlib import Path
import sys, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
REPORTS = ROOT / 'outputs' / 'reports'
FIGS    = ROOT / 'outputs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / 'src'))

# Load the current development-split result CSVs
# Load all method result CSVs
_method_files = [
    ('b1', 'baseline_b1_dev.csv'), ('b2', 'baseline_b2_dev.csv'),
    ('b3', 'baseline_b3_dev.csv'), ('mm', 'pipeline_dev.csv'),
]
b1, b2, b3, mm = [
    pd.read_csv(REPORTS / f).drop_duplicates('qid', keep='last')
    for _, f in _method_files
]

METHODS = {
    'B1\nZero-shot':    b1,
    'B2\nFew-shot':     b2,
    'B3\nDIN-SQL':      b3,
    'Constrained pipeline\n(ours)': mm,
}
COLORS  = ['#d73027', '#fc8d59', '#4575b4', '#1a9850']
LABELS  = list(METHODS.keys())
DFS     = list(METHODS.values())

print('Loaded results:')
for lbl, df in METHODS.items():
    lbl_clean = lbl.replace('\n', ' ')
    print(f'  {lbl_clean:<20} n={len(df):<3} ExecRate={df["exec_ok"].mean():.1%}  DenAcc={df["den_acc"].mean():.1%}  Lat={df["latency_s"].mean():.1f}s')


## 1. Overall Comparison Table

The primary result: execution success rate and denotation accuracy across all four methods.

**Denotation accuracy** (DenAcc) measures whether the predicted SQL returns the same result set
as the gold SQL â€" the strictest correctness criterion, following BIRD (Li et al., 2023).
**Execution rate** (ExecRate) measures whether the predicted SQL runs without error.
A high ExecRate with low DenAcc indicates syntactically valid but semantically wrong SQL.

In [ ]:
# Print summary table
print(f"{'Method':<18} {'ExecRate':>10} {'DenAcc':>9} {'JoinHall':>10} {'AvgLat':>8}")
print('-' * 58)
for lbl, df in METHODS.items():
    lbl_c = lbl.replace('\n', ' ')
    er  = df['exec_ok'].mean()
    da  = df['den_acc'].mean()
    jh  = df['join_hall'].mean() if 'join_hall' in df.columns else 0.0
    lat = df['latency_s'].mean()
    print(f"{lbl_c:<18} {er:>10.1%} {da:>9.1%} {jh:>10.1%} {lat:>7.1f}s")

summary = []
for lbl, df in METHODS.items():
    summary.append({
        'method': lbl.replace('\n', ' '),
        'split': 'dev',
        'n': len(df),
        'exec_rate': round(df['exec_ok'].mean(), 4),
        'den_acc': round(df['den_acc'].mean(), 4),
        'join_hall': round(df['join_hall'].mean(), 4) if 'join_hall' in df.columns else 0.0,
        'avg_lat_s': round(df['latency_s'].mean(), 2),
    })

pd.DataFrame(summary).to_csv(REPORTS / 'dev_summary.csv', index=False)
print('\nSaved: outputs/reports/dev_summary.csv')


## 2. Main Comparison Bar Chart

Two metrics side by side: ExecRate and DenAcc per method.
This figure now reflects the **74-question development split** on the expanded benchmark.

**Interpretation (dev set, n=74):**
- Constrained pipeline is the strongest system on denotation accuracy (64.9%), ahead of B1 (41.9%), B3 DIN-SQL (40.5%), and B2 Few-shot (23.0%).
- B1 remains fast but unsafe: it is the only baseline with non-zero join hallucination on the dev split.
- B2 Few-shot executes almost everything but spends far more latency for much lower accuracy than Constrained pipeline.
- B3 DIN-SQL is more structured than the prompt-only baselines but still lags Constrained pipeline on both accuracy and latency.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x       = np.arange(len(LABELS))
w       = 0.35
er_vals = [df["exec_ok"].mean() for df in DFS]
da_vals = [df["den_acc"].mean()  for df in DFS]

bars1 = ax.bar(x - w/2, er_vals, w, label="Execution Rate",  color="#4575b4", alpha=0.85)
bars2 = ax.bar(x + w/2, da_vals, w, label="Denotation Accuracy", color="#1a9850", alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.0%}", ha="center", va="bottom", fontsize=9, fontweight="bold")
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.0%}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(LABELS, fontsize=10)
ax.set_ylabel("Rate", fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_title("NL-to-SQL Performance on SAP O2C OCEL Dev Set (n=74)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.grid(axis="y", alpha=0.3)
ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
for ext in ("pdf","png"):
    plt.savefig(FIGS / f"comparison_bar.{ext}", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: comparison_bar.pdf/png")


## 3. DenAcc by Query Class

Breaks down accuracy per query class to show where each method succeeds and fails.
This reveals which query types drive the performance gap between methods.

**Interpretation:**
- `count_filter` remains simple enough that all schema-aware methods are strong.
- `temporal_trend`: 91.7% DenAcc on dev, 66.7% on test — strong dev performance; test gap warrants continued attention.
- `group_topk`: stable on dev but weaker on test, suggesting aggregation wording is still brittle.
- `anomaly_filter`: still the hardest class; Constrained pipeline reaches 50%, with remaining misses driven by relation-type selection and multi-step delay reasoning.
- `path_relation`: the verifier removes invalid joins, but denotation still depends on choosing the correct whitelisted relation.
- `delay_analysis`: improved on dev and test, but still depends on longer CTE-style delay constructions.

In [ ]:
classes  = sorted(b1["query_class"].unique())
n_cls    = len(classes)
x        = np.arange(n_cls)
w        = 0.2

fig, ax = plt.subplots(figsize=(13, 5))
for i, (lbl, df, col) in enumerate(zip(LABELS, DFS, COLORS)):
    vals = [df[df["query_class"]==c]["den_acc"].mean() for c in classes]
    offset = (i - 1.5) * w
    bars = ax.bar(x + offset, vals, w, label=lbl.replace("\n"," "), color=col, alpha=0.85)
    for bar, v in zip(bars, vals):
        if v > 0.05:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{v:.0%}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([c.replace("_","-") for c in classes], fontsize=9)
ax.set_ylabel("Denotation Accuracy", fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title("Denotation Accuracy by Query Class", fontsize=12, fontweight="bold")
ax.legend(fontsize=9, ncol=4)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.grid(axis="y", alpha=0.3)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
for ext in ("pdf","png"):
    plt.savefig(FIGS / f"denacc_by_class.{ext}", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: denacc_by_class.pdf/png")


## 4. DenAcc by Difficulty

Shows accuracy across easy / medium / hard questions per method.

**Interpretation:**
- All methods degrade on hard questions â€" multi-hop CTEs and complex joins are harder.
- B3 and B2 maintain ~50% on hard questions â€" schema context and multi-step decomposition help.
- Constrained pipeline recovers on hard via the repair loop and 3-CTE compiler for delay/anomaly queries.
- Easy questions: B3/B2/Method-M all achieve 90%+, confirming schema-prompting lift on simple queries.


In [ ]:
diffs   = ["easy", "medium", "hard"]
x       = np.arange(len(diffs))
w       = 0.2

fig, ax = plt.subplots(figsize=(9, 5))
for i, (lbl, df, col) in enumerate(zip(LABELS, DFS, COLORS)):
    vals = [df[df["difficulty"]==d]["den_acc"].mean() for d in diffs]
    offset = (i - 1.5) * w
    bars = ax.bar(x + offset, vals, w, label=lbl.replace("\n"," "), color=col, alpha=0.85)
    for bar, v in zip(bars, vals):
        if v > 0.05:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{v:.0%}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(diffs, fontsize=11)
ax.set_ylabel("Denotation Accuracy", fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title("Denotation Accuracy by Difficulty Level", fontsize=12, fontweight="bold")
ax.legend(fontsize=9, ncol=4)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.grid(axis="y", alpha=0.3)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
for ext in ("pdf","png"):
    plt.savefig(FIGS / f"denacc_by_difficulty.{ext}", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: denacc_by_difficulty.pdf/png")


## 5. Constrained pipeline - Execution Status Breakdown

Shows how the constrained pipeline terminates on the current 74-question dev run.
This is critical for diagnosing residual weaknesses.

**Interpretation:**
- Most dev questions are accepted on the first attempt; a smaller subset is rescued after repair.
- The remaining failures are concentrated in safe rejection or execution-side issues, not in uncontrolled join invention.
- This status view complements denotation accuracy: it shows where the verifier is trading coverage for structural safety.


In [ ]:
status_plot = mm.copy()
status_plot['status_display'] = np.select(
    [
        (status_plot['status'] == 'accept') & (status_plot['ir_attempts'].astype(float) <= 1),
        (status_plot['status'] == 'accept') & (status_plot['ir_attempts'].astype(float) > 1),
        status_plot['status'] == 'reject',
        status_plot['status'] == 'exec_error',
    ],
    ['accept first', 'accept after repair', 'reject', 'exec. error'],
    default=status_plot['status']
)
status_order = ['accept first', 'accept after repair', 'reject', 'exec. error']
status_colors = {
    'accept first': '#1a9850',
    'accept after repair': '#74add1',
    'reject': '#984ea3',
    'exec. error': '#d73027',
}
status_counts = status_plot['status_display'].value_counts().reindex(status_order, fill_value=0)
labels_pie = [s for s in status_order if status_counts[s] > 0]
sizes = [int(status_counts[s]) for s in labels_pie]
pie_colors = [status_colors[s] for s in labels_pie]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.8, 5.2), gridspec_kw={'width_ratios': [0.9, 1.35]})

wedges, texts, autotexts = ax1.pie(
    sizes, labels=None, autopct='%1.1f%%', startangle=90,
    colors=pie_colors, pctdistance=0.74,
    wedgeprops=dict(width=0.48, edgecolor='white', linewidth=2))
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight('bold')
ax1.legend(wedges, [f'{l} ({s})' for l, s in zip(labels_pie, sizes)],
           loc='upper center', bbox_to_anchor=(0.5, -0.05), fontsize=9,
           ncol=1, frameon=False)
ax1.set_title('Pipeline Status\nConstrained pipeline, dev split (n=74)', fontsize=12, fontweight='bold')

classes_list = sorted(status_plot['query_class'].unique())
y = np.arange(len(classes_list))
left = np.zeros(len(classes_list))
for st in status_order:
    vals = []
    for c in classes_list:
        class_rows = status_plot[status_plot['query_class'] == c]
        vals.append((class_rows['status_display'] == st).sum() / max(len(class_rows), 1))
    if any(v > 0 for v in vals):
        ax2.barh(y, vals, left=left, label=st, color=status_colors[st], alpha=0.88, edgecolor='white')
    left += np.array(vals)
ax2.set_yticks(y)
ax2.set_yticklabels([c.replace('_', '-') for c in classes_list], fontsize=9)
ax2.set_xlabel('Proportion within query class', fontsize=10)
ax2.set_xlim(0, 1.0)
ax2.set_title('Status by Query Class', fontsize=12, fontweight='bold')
ax2.grid(axis='x', alpha=0.28)
ax2.spines[['top', 'right']].set_visible(False)
ax2.legend(loc='lower right', fontsize=8, frameon=False)

fig.subplots_adjust(left=0.06, right=0.98, bottom=0.24, top=0.86, wspace=0.30)
for ext in ('pdf', 'png'):
    plt.savefig(FIGS / f'pipeline_status.{ext}', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: pipeline_status.pdf/png')


## 6. Latency Comparison

Average query translation latency per method (LLM calls only; DuckDB execution <0.1s).

**Interpretation:**
- B1 (2.1s) is the fastest method, but it gives much lower denotation accuracy and is the only system with non-zero join hallucination on dev.
- Constrained pipeline (3.3s) remains interactive while adding schema retrieval, verification, compilation, and bounded repair.
- B3 DIN-SQL (10.0s) is slower because it uses staged decomposition and self-correction prompts.
- B2 Few-shot (65.0s) is slowest because it evaluates multiple shots and votes by result hash.


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
lats = [df["latency_s"].mean() for df in DFS]
bars = ax.bar(range(len(LABELS)), lats, color=COLORS, alpha=0.85, edgecolor="white", linewidth=0.5)
for bar, v in zip(bars, lats):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f"{v:.1f}s", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_xticks(range(len(LABELS)))
ax.set_xticklabels(LABELS, fontsize=10)
ax.set_ylabel("Avg latency (seconds)", fontsize=11)
ax.set_title("Average Query Translation Latency", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
for ext in ("pdf","png"):
    plt.savefig(FIGS / f"latency.{ext}", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: latency.pdf/png")


## 7. Join Hallucination Rate

Measures how often each method produces SQL referencing joins not in the relation whitelist.
This is the core **safety metric**: hallucinated joins can return plausible numbers while still misrepresenting the business process.

**Interpretation:**
- B1 remains the only method with non-zero join hallucination on the dev split.
- B2 Few-shot and B3 DIN-SQL reach 0% join hallucination empirically on this benchmark, but without a structural guarantee.
- Constrained pipeline reaches 0% **by construction** because the verifier blocks invalid relation paths before SQL compilation.


In [ ]:
methods_jh  = ['B1\nZero-shot', 'B2\nFew-shot', 'B3\nDIN-SQL', 'Constrained pipeline\n(ours)']
dfs_jh      = [b1, b2, b3, mm]
colors_jh   = ['#d73027','#fc8d59','#4575b4','#1a9850']

jh_vals = []
for df in dfs_jh:
    if 'join_hall' in df.columns:
        jh_vals.append(df['join_hall'].mean())
    else:
        jh_vals.append(0.0)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(range(len(methods_jh)), jh_vals, color=colors_jh, alpha=0.85, edgecolor='white', linewidth=0.5)
for bar, v in zip(bars, jh_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f'{v:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(range(len(methods_jh)))
ax.set_xticklabels(methods_jh, fontsize=10)
ax.set_ylabel('Join Hallucination Rate', fontsize=11)
ax.set_ylim(0, 0.20)
ax.set_title('Join Hallucination Rate by Method\n(Constrained pipeline: 0% by verifier guarantee)', fontsize=11, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
for ext in ('pdf','png'):
    plt.savefig(FIGS / f'join_hallucination.{ext}', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: join_hallucination.pdf/png')


## 8. Key Findings Summary

Results from the current development split (n=74) and held-out test split (n=46). Backend: DeepSeek-Chat.

### Dev set

**Correctness:** Constrained pipeline leads the dev split at 64.9% denotation accuracy, ahead of B1 Zero-shot (41.9%), B3 DIN-SQL (40.5%), and B2 Few-shot (23.0%).

**Safety:** Constrained pipeline keeps join hallucination at 0% by construction. B2 and B3 also record 0% empirically on this benchmark, but without a verifier guarantee.

**Grounding:** Constrained pipeline returns typed IR, compiled SQL, and auditable provenance metadata; the baselines return SQL only.

**Efficiency:** Constrained pipeline (3.3s) is much faster than B2 Few-shot (65.0s) and B3 DIN-SQL (10.0s), while still leading on denotation accuracy.

**Robustness:** Constrained pipeline is strongest on temporal-trend and path-relation questions, while anomaly, conformance, and window-style questions remain the hardest.

### Test set (held-out, n=46)

| Method | ExecRate | DenAcc | JoinHall |
| ------ | -------- | ------ | -------- |
| B1 Zero-shot | 93.5% | 41.3% | 0.0% |
| B2 Few-shot | 97.8% | 21.7% | 0.0% |
| B3 DIN-SQL | 73.9% | 45.7% | 0.0% |
| **Constrained pipeline** | **78.3%** | **50.0%** | **0.0%** |

Constrained pipeline remains the top-accuracy system on the held-out split, but the margin over B3 DIN-SQL narrows to 4.3 percentage points, so the test result should be read as corroborative rather than final separation.

### Limitations

- Window aggregation, anomaly filtering, and some conformance patterns remain the main weak spots.
- Prompt-based baselines and Constrained pipeline were all run on the same backend, which isolates pipeline effects but does not replace broader cross-model evaluation.


In [ ]:
# Export summary table for LaTeX
latex_rows = []
for lbl, df in METHODS.items():
    lbl_c = lbl.replace("\n", " ")
    er  = df["exec_ok"].mean()
    da  = df["den_acc"].mean()
    jh  = df["join_hall"].mean() if "join_hall" in df.columns else 0.0
    lat = df["latency_s"].mean()
    latex_rows.append(f"{lbl_c} & {er:.1%} & {da:.1%} & {jh:.1%} & {lat:.1f}s \\\\")

print("LaTeX table rows (paste into paper):")
print("\\hline")
for row in latex_rows:
    print(row)
print("\\hline")
print()
print("All figures saved to outputs/figures/*.pdf")
print("evaluation evaluation complete.")


## 9. Repair Loop Analysis (Verifier Effectiveness)

The verifier runs up to 2 repair attempts before rejecting. This section quantifies when repair is needed and whether it helps.

**Interpretation:**  
- Questions accepted on first attempt (ir_attempts=1) have higher DenAcc than repaired questions - not because repair degrades, but because questions requiring repair are structurally harder (anomaly_filter, delay_analysis with missing joins).  
- The repair loop prevents *structural* failures (missing threshold, wrong column) but cannot correct *semantic* errors (wrong relation type).


In [ ]:
# Repair loop analysis
dev_split = mm.copy()
dev_split["repaired"] = dev_split["ir_attempts"] > 1

print("=== Dev set repair breakdown ===")
rb = dev_split.groupby("repaired")["den_acc"].agg(n="count", DenAcc="mean")
rb.index = ["First attempt (ir_attempts=1)", "After repair (ir_attempts>1)"]
print(rb.to_string())

print()
print("=== Per-class repair breakdown (dev) ===")
cls_tbl = []
for cls in sorted(dev_split["query_class"].unique()):
    sub = dev_split[dev_split["query_class"] == cls]
    n = len(sub)
    first = (sub["ir_attempts"] == 1).sum()
    repaired = (sub["ir_attempts"] > 1).sum()
    rejected = (sub["status"] == "reject").sum()
    acc = sub["den_acc"].mean()
    cls_tbl.append({"class": cls, "n": n, "first_try": first, "repaired": repaired, "rejected": rejected, "DenAcc": f"{acc:.0%}"})

import pandas as pd
cls_df = pd.DataFrame(cls_tbl)
print(cls_df.to_string(index=False))

# Count repair outcomes
print()
repair_rows = dev_split[dev_split["repaired"]]
print(f"Repaired questions: {len(repair_rows)}  ->  DenAcc after repair: {repair_rows['den_acc'].mean():.0%}")
reject_rows = dev_split[dev_split["status"] == "reject"]
print(f"Rejected (all repairs exhausted): {len(reject_rows)}")


## 10. Failure Taxonomy - Constrained pipeline (Class Robustness)

Classifying a 12-failure sample (7 dev + 5 test) by root cause, so the robustness discussion rests on evidence rather than anecdote. This is a hand-picked subset, not the full set: the pipeline has 26 failures on dev and 23 on test (49 total), and the sample spans 5 of the 9 query classes. It contains no `window_agg` questions, so it cannot explain the weakest class.

| Category | Description |
|---|---|
| `wrong_event_type` | Hallucinated/aliased event type; verifier should catch post-enum fix |
| `wrong_table` | Counted from events instead of objects (or vice versa) |
| `anti_join` | NOT-linked / exclusion logic error in path query |
| `multi_hop` | Multi-hop join produces wrong aggregation result |
| `wrong_relation` | Correct intent, wrong relation type chosen semantically |
| `compile_error` | IR valid, SQL compiler table alias bug |
| `reject_repair_fail` | Verifier rejected; both repair attempts failed |
| `column_hallucination` | Hallucinated column caught by verifier; repair also wrong |


In [ ]:
taxonomy = [
    # DEV failures
    {"qid": "Q005", "split": "dev",  "class": "count_filter",  "difficulty": "easy",   "status": "accept", "den_acc": 0,
     "category": "wrong_event_type",    "note": "'invoice creation' -> hallucinated; should be billing_created"},
    {"qid": "Q034", "split": "dev",  "class": "count_filter",  "difficulty": "easy",   "status": "accept", "den_acc": 0,
     "category": "wrong_table",         "note": "counted from events; distinct customers should query objects"},
    {"qid": "Q046", "split": "dev",  "class": "path_relation", "difficulty": "medium", "status": "accept", "den_acc": 0,
     "category": "anti_join",           "note": "NOT-linked logic (never delivered) produces wrong count"},
    {"qid": "Q047", "split": "dev",  "class": "path_relation", "difficulty": "hard",   "status": "accept", "den_acc": 0,
     "category": "multi_hop",           "note": "order_item -> customer -> dunning multi-hop join fails"},
    {"qid": "Q048", "split": "dev",  "class": "path_relation", "difficulty": "medium", "status": "exec_error", "den_acc": 0,
     "category": "compile_error",       "note": "GROUP BY references raw table alias 'order_item' not available in join view"},
    {"qid": "Q027", "split": "dev",  "class": "delay_analysis","difficulty": "hard",   "status": "reject",     "den_acc": 0,
     "category": "reject_repair_fail",  "note": "'invoice creation' ambiguous; LLM failed to add join in 2 repair attempts"},
    {"qid": "Q029", "split": "dev",  "class": "anomaly_filter","difficulty": "hard",   "status": "accept", "den_acc": 0,
     "category": "wrong_relation",      "note": "chose order_to_billing; 'clearing gaps' requires billing_to_ar"},
    # TEST failures
    {"qid": "Q009", "split": "test", "class": "group_topk",    "difficulty": "easy",   "status": "accept", "den_acc": 0,
     "category": "multi_hop",           "note": "top event types - ORDER BY / LIMIT combo wrong"},
    {"qid": "Q014", "split": "test", "class": "path_relation", "difficulty": "medium", "status": "accept", "den_acc": 0,
     "category": "multi_hop",           "note": "relation counts per type - aggregation grouping error"},
    {"qid": "Q035", "split": "test", "class": "count_filter",  "difficulty": "easy",   "status": "accept", "den_acc": 0,
     "category": "wrong_table",         "note": "unique materials: should count objects not events"},
    {"qid": "Q043", "split": "test", "class": "anomaly_filter","difficulty": "hard",   "status": "accept", "den_acc": 0,
     "category": "wrong_relation",      "note": "dunning notice - chose order_to_customer; correct is dunning_sent"},
    {"qid": "Q044", "split": "test", "class": "group_topk",    "difficulty": "medium", "status": "reject",     "den_acc": 0,
     "category": "column_hallucination","note": "attributes->>'currency' not in schema; verifier caught; repair also wrong"},
]

import pandas as pd
tax_df = pd.DataFrame(taxonomy)
print("=== Failure Taxonomy (all 12 Constrained pipeline failures) ===")
print(tax_df[["qid","split","class","difficulty","category","note"]].to_string(index=False))

print()
print("=== Category summary ===")
cat_summary = tax_df["category"].value_counts().reset_index()
cat_summary.columns = ["category", "count"]
print(cat_summary.to_string(index=False))

print()
print("=== Failures by class ===")
fc = tax_df.groupby("class")["qid"].count().sort_values(ascending=False)
print(fc.to_string())
print()
print("Key finding: wrong_relation (n=2) and multi_hop (n=3) account for 42% of failures.")
print("Both are principal residual weaknesses: wrong-relation-type selection and multi-hop join aggregation.")
